In [1]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial import distance
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [108]:
n_community = 2
n_members = 3

tokens = []

for ii in range(n_community*n_members+1):
    tokens.append(
        chr(ord('A')+ii)
    )

In [109]:
class brain(nn.Module):
    def __init__(self, input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers=2, num_layers_sleep=2):
        super(brain, self).__init__()

        self.rnn = nn.RNN(input_size+sleep_output_size, hidden_wake_size, num_layers, nonlinearity='relu', batch_first=True)
        self.sleep_rnn = nn.RNN(1, hidden_sleep_size, num_layers_sleep, nonlinearity='relu', batch_first=True)
        self.sleep_fc = nn.Linear(hidden_sleep_size, sleep_output_size)
        # self.fc = nn.Linear(hidden_wake_size, 15)
        self.wake_fc = nn.Linear(hidden_wake_size, len(tokens))
        self.sleep_output_size = sleep_output_size

    def forward(self, x, x_=None, hw=None, hs=None, sleep=False):
        # print(x.shape, 'x')
        if sleep:
            if hs == None:
                out, hs = self.sleep_rnn(x_)
            else:
                out, hs = self.sleep_rnn(x_, hs)
            # print(out.shape)
            sleep_out = self.sleep_fc(out)
        else:
            sleep_out = torch.zeros((1,x.size(1),self.sleep_output_size))
            
        # print(x.size())
        x = torch.cat((x,sleep_out), dim=2)
        
        if hw == None:
            out, hw = self.rnn(x)
        else:
            out, hw = self.rnn(x, hw)

        # out_ = self.fc(out) 
        out = self.wake_fc(out[:,-1,:])

        if sleep:
            return out, hw, hs
        else:
            return out, hw

In [110]:
def compute_geodesic(hidden1, hidden2):

    total_layers = len(hidden1)
    w = 0

    for ii in range(total_layers):
        w_ = np.array(dist( hidden1[ii], hidden2[ii], 'cosine'))
        w += w_
           
    return w[0][0]/total_layers

In [111]:
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=8):
        
        one_hot_encoded = np.zeros((len(data), len(tokens)), dtype=float)
        for ii, token in enumerate(data):
            one_hot_encoded[ii,ord(token)-65] = 1
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory, len(tokens)*working_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), len(tokens)))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                for kk in range(working_memory):
                    self.X[ii,jj,kk*len(tokens):(kk+1)*len(tokens)] = \
                    one_hot_encoded[ii+jj+kk,:]
                    
            self.y[ii] = \
                one_hot_encoded[ii+jj+kk+1,:]

        self.X = tnsr(self.X).float()
        self.y = tnsr(self.y).float()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [147]:
### initial training ###
total_samples = 50000
working_memory = 1
short_term_memory = 2
hidden_wake_size = 30
hidden_sleep_size = 10
sleep_output_size = 5
num_layers_wake = 1
num_layers_sleep = 1
output_sleep = len(tokens)
input_size = len(tokens)*working_memory
lr = 4e-4
test_acc = []

data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

network1 = brain(input_size, hidden_wake_size, hidden_sleep_size, sleep_output_size, num_layers_wake, num_layers_sleep)

optimizer = torch.optim.SGD(network1.parameters(), lr=lr, momentum=0.95)
criterion = torch.nn.CrossEntropyLoss()

total = 0
correct = np.zeros(1000,dtype=float)
for X, y in train_loader:
    optimizer.zero_grad()

    if total == 0:
        predicted_y, hidden = network1(X)
    else:
        predicted_y, hidden = network1(X, hw=mem)
    
    # print(predicted_y.shape, y.shape)
    loss = criterion(predicted_y, y)
    loss.backward(retain_graph=True)
    optimizer.step()

    with torch.no_grad():
        mem=hidden.clone()
        true_y = y.argmax(axis=1)
        estimated_y = predicted_y.argmax(axis=1)

        total += 1
        if true_y == estimated_y:
                correct[total%1000] = 1
        else:
            correct[total%1000] = 0

        test_acc.append(
            np.sum(correct)/total if total<1000 else np.sum(correct)/1000
        )
        if total%1000 == 0:
            print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc[-1]:.4f}')

Iter : 1001, loss: 2.1591, accuracy: 0.2410
Iter : 2001, loss: 1.6309, accuracy: 0.2680
Iter : 3001, loss: 1.3236, accuracy: 0.4340
Iter : 4001, loss: 1.3135, accuracy: 0.5570
Iter : 5001, loss: 1.4285, accuracy: 0.5650
Iter : 6001, loss: 0.7805, accuracy: 0.6080
Iter : 7001, loss: 0.5499, accuracy: 0.6600
Iter : 8001, loss: 0.5992, accuracy: 0.6520
Iter : 9001, loss: 0.8888, accuracy: 0.6580
Iter : 10001, loss: 0.4923, accuracy: 0.6690
Iter : 11001, loss: 0.9411, accuracy: 0.6690
Iter : 12001, loss: 0.9189, accuracy: 0.6670
Iter : 13001, loss: 1.0284, accuracy: 0.6540
Iter : 14001, loss: 0.5223, accuracy: 0.6810
Iter : 15001, loss: 0.3483, accuracy: 0.6640
Iter : 16001, loss: 0.5056, accuracy: 0.6670
Iter : 17001, loss: 0.5243, accuracy: 0.6670
Iter : 18001, loss: 0.6132, accuracy: 0.6580
Iter : 19001, loss: 0.8717, accuracy: 0.6730
Iter : 20001, loss: 0.7561, accuracy: 0.6620
Iter : 21001, loss: 0.3758, accuracy: 0.6730
Iter : 22001, loss: 0.9608, accuracy: 0.6680
Iter : 23001, loss:

# RNN equation

$h_{t+1} = \phi (W_{hh} h_t + W_{xh} x_t + b)$ 

# Attractor network equation

$h_{t+1} = \phi (W_{hh} h_t + \text{externel input to the neuron})$

In [148]:
centroids = []

total_samples = 10000
steps = 10
idx = torch.randint(0, len(tokens), (1,)) [0]
X_hat = torch.zeros(len(tokens),dtype=torch.float32)
X_hat[idx] = 1.0
counts = []
seq = ''

for jj in tqdm(range(total_samples)):
    with torch.no_grad():
        # print(tokens[idx])
        if jj == 0:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1))
        else:
            X_hat_, hidden_state = network1(X_hat.reshape(1,1,-1), hidden_state)

        
        mem = hidden_state.clone() 
        for ii in range(steps):
            _, mem = network1(X_hat.reshape(1,1,-1), mem)
            # print(torch.abs(torch.sum(mem-mem_)))

        hidden_state = mem
        
        flag = True
        for center in centroids:
            # print(torch.abs(torch.sum(mem-center)))
            if torch.abs(torch.sum(mem-center)).detach().numpy() < 1e-2:
                flag = False
                break
        
        if flag:
            centroids.append(mem)

        X_hat = torch.nn.functional.softmax(X_hat_, dim=1)
        dist_categ = torch.distributions.Categorical(probs=X_hat.reshape(-1))
        idx = dist_categ.sample()

        X_hat = torch.zeros(len(tokens),dtype=torch.float32)
        X_hat[idx] = 1.0
        X_hat = X_hat.reshape(1,1,-1)
        seq = seq + tokens[idx]
        # print('\n')
        

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:03<00:00, 2516.11it/s]


In [149]:
len(centroids)

7

In [150]:
centroids

[tensor([[[0.0948, 0.0000, 0.0800, 0.2935, 0.9033, 0.3427, 1.0871, 0.4307,
           0.0000, 0.4112, 0.0000, 0.0000, 0.0000, 0.0000, 0.7374, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.1577, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.1050]]]),
 tensor([[[0.0000, 0.0000, 1.1237, 1.2766, 0.0000, 0.0000, 0.0829, 0.5133,
           0.0000, 0.0154, 0.0000, 1.0470, 0.0000, 0.5565, 0.9007, 0.1463,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.8803, 0.0000, 0.0000,
           0.0000, 1.0026, 0.0000, 0.0000, 0.0000, 0.6885]]]),
 tensor([[[0.0000, 0.0000, 0.0000, 1.1926, 0.0000, 0.0000, 0.0542, 0.4073,
           0.9447, 1.1210, 0.0319, 0.0000, 0.0000, 0.5838, 0.0000, 0.0000,
           0.0000, 0.0000, 0.3329, 0.0000, 0.0000, 1.1919, 0.0000, 0.0000,
           0.5239, 0.1638, 0.0000, 0.0000, 0.0000, 1.4749]]]),
 tensor([[[0.1168, 0.0000, 0.0000, 0.0000, 0.0000, 0.1038, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.0314, 0.

In [151]:
for center in centroids:
    for center_ in centroids:
        print(distance.cosine(center[0][0].detach().numpy(), center_[0][0].detach().numpy()))

    print('\n')

0.0
0.51060396
0.40162247
0.48322386
0.7918656
0.6328443
0.5952188


0.51060396
0.0
0.45271093
0.56511295
0.72740126
0.70404077
0.42991626


0.40162247
0.45271093
0.0
0.41001862
0.8541489
0.53662515
0.6518051


0.48322386
0.56511295
0.41001862
0.0
0.6152804
0.6369734
0.48117065


0.7918656
0.72740126
0.8541489
0.6152804
0.0
0.54202294
0.5065197


0.6328443
0.70404077
0.53662515
0.6369734
0.54202294
0.0
0.47309119


0.5952188
0.42991626
0.6518051
0.48117065
0.5065197
0.47309119
0.0




In [152]:
graph = {}

for ii, center in enumerate(centroids):
    graph[ii] = []
    min_dis = 3
    min_id = -1
    for jj, center_ in enumerate(centroids):
        if ii != jj:
            dis = distance.cosine(center[0][0].detach().numpy(), center_[0][0].detach().numpy())

            if dis < min_dis:
                min_dis = dis
                min_id = jj
    graph[ii].append(min_id)

In [153]:
graph

{0: [2], 1: [6], 2: [0], 3: [2], 4: [6], 5: [6], 6: [1]}

In [154]:
n_samples = 1000
data = get_sequence(total_samples, n_community, n_members, train_percent=1.0)

data_set = Dataset_converter(data, working_memory, short_term_memory)
train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

community = {}
for ii in range(len(centroids)):
    community[ii] = []

ii = 0

for X, y in train_loader:
    if ii == 0:
        # seq += tokens[idx]        
        X_hat, mem = network1(X)
        # centroids.append(mem[0][0])
    else:
        X_hat, mem = network1(X, mem)
    
    dis = []
    for center in centroids:
        dis.append(
            distance.cosine(center[0][0].detach().numpy(), mem[0][0].detach().numpy())
        )
    
    idx = int(np.argmin(dis))
    community[idx].append(tokens[X[0][-1].argmax()])

         
    ii += 1

In [155]:
for com in community.keys():
    print(np.unique(community[com]))

['F']
['D']
['E']
['G']
['B']
['A']
['C']


In [156]:
def find_connected_components(graph):
    visited = set()
    components = []

    # Symmetrize the graph (make it undirected)
    undirected_graph = {}
    for node, neighbors in graph.items():
        for neighbor in neighbors:
            undirected_graph.setdefault(node, []).append(neighbor)
            undirected_graph.setdefault(neighbor, []).append(node)

    def dfs(node, component):
        visited.add(node)
        component.append(node)
        for neighbor in undirected_graph.get(node, []):
            if neighbor not in visited:
                dfs(neighbor, component)

    for node in undirected_graph:
        if node not in visited:
            component = []
            dfs(node, component)
            components.append(component)
    
    return components

In [157]:
components = find_connected_components(graph)
print("Connected Components:", components)

Connected Components: [[0, 2, 3], [1, 6, 4, 5]]


In [92]:
data

'DABCIHEFGIGHEFIBADCIHEFGIHEFGIGFEHIHGFEIHGFEIFEHGIFEHGIBADCIBCDAIFEHGIDABCIABCDIDCBAIDCBAIADCBIGFEHIABCDIFGHEIHEFGIBADCIABCDIGFEHIGHEFIHGFEIEHGFIBADCIFGHEICDABIHEFGIABCDIFGHEIGHEFIDCBAIGHEFIHEFGIADCBICDABIFEHGIDABCIGHEFIHEFGIGFEHIGFEHIEHGFIBADCIGHEFIABCDIHEFGIHEFGIGFEHIFEHGIDCBAIGHEFIDABCIABCDIGFEHIGHEFICBADIEFGHICDABIGHEFIABCDIDABCIADCBIBADCIADCBIHGFEIGHEFICBADIABCDIBADCIEHGFIFGHEIADCBIFGHEIBCDAICDABIFEHGICDABIEFGHIEFGHIEHGFIEHGFIEHGFIGFEHIADCBIFGHEIHEFGICBADIBCDAIHGFEIDABCICDABIFEHGIHEFGIDCBAIGHEFIABCDIDABCIGFEHICDABIBCDAICBADIGFEHIEFGHIDCBAIFGHEICDABIBCDAIDCBAICBADICBADIFEHGIABCDIHEFGIDABCIHEFGIGHEFIGFEHIGFEHIFEHGIDCBAIHEFGIHEFGIEHGFIGFEHIDCBAIHEFGIGHEFICBADIDABCIFEHGIBCDAIBCDAIBADCIADCBIFEHGIHEFGIDCBAIDABCIFEHGICDABIHEFGIFGHEIEHGFIEHGFIDCBAICDABIADCBIFEHGIBCDAIEFGHIHEFGIHGFEIADCBIFGHEICDABIBCDAIGFEHICDABIEFGHIHEFGICBADIBCDAIHGFEIDABCIFGHEIABCDIHEFGICDABIFGHEICDABIFGHEIBCDAIDABCIBADCIHGFEIBCDAIHEFGIDABCIEFGHIDABCIDABCIDCBAIADCBICBADIGFEHIABCDIDABCIHGFEIBCDAICDABIDCBAIHGFEICDABIDABC

In [91]:
seq[-1000:]

'BCDCDCDADCDADCDADABCBADCDCDCDADADCDCDCBCDCDCDADCDABCDCBCBCBABCDADCBCBADCDADCDCDADCDABADABCBCDHGFEHEHEFGHEFHEFEFGFEFGFEFGHGFGFGFEFEFEFGFEFEHEHGHGHEFEFGCBADADCDCDADADCDCDCDCDADCDADADCDADADCDCDADCDCDCDADCDCDCDCBCDADADCDCBADCBCDCDCBCDADADCDCDCGHGHGHEHEFGHEFEFEFEFDADCDADFEFDCDCDCDCDCCDADCDADCDADCDCDCDABCBCDABCDADCBADCDABCDAHEHEFEFGFEFEFEFGFEFGFGHGFDEFGFEFGFEHEFEHGFEHGFEFGFGFEHGFEHEHGFGFGFEFEFGHEHCDCDCDCDABCDADCBCBCBCBCDCBCGHGFEFEHEFEHGFGFGFEFEHEFGFEHEFGHGFGFEBCDCHEFGFEFEFEFEFGFEFGHEFGHGFEFEFEFEHEFGFEFGFHGHGHGFHGFGFGHEFEFEFGFEFEFEHGHGFGFEFEHEHEHGFGFEFHGFEFGFGHGFEFGHGFEFGHGFHGFEHEFGHEHEFEFEHGHGFEFEFGFEHEFEFEFGFEFEFEFEHEHEFEFEFEFGFEFEHGFEFGFEFEHEFEFGFEFGFEHEFGFGFGFEFEFGFGHGFEFEFGFEFGFGHEFECBCDCBABADCDCDCDCBADADCBABABADCDCBCDCBCBCDCBCDCDCDCDCBCDADCDCDCBCDCDCDCDCDADCDADCDCDABCDCBCBDEFGFGFEFEHGHFEFEFEFEFGCDCBCGFGFEFEFEFEFGFEHEHGFGFEHEFGHGHGFEFEHEFGFGFEHGHGFEHGHEHEFEFGFGFGFEFEFEFGHEFEHEHEHEHEHEFEFEHGFEFEFECDCDABCBADCBCDADCFEFEFEFEFEHEFGFGHEFGHEHEHGFEFGFEFGHGFGFEFGHEHGHEFEFEFEFEFEFEFEFEFGHEFEFEHGH

In [65]:
import numpy as np
from gng import GrowingNeuralGas
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# ------------------------------
# Simulated hidden states
# Replace this with actual RNN outputs (e.g., from LSTM)
# ------------------------------
np.random.seed(0)
hidden_states = []

for center in centroids:
    hidden_states.append(
        center[0][0].detach().numpy()
    )

# Optional: normalize for cosine-like distance behavior
hidden_states_normalized = normalize(hidden_states, axis=1)

# ------------------------------
# Train Growing Neural Gas
# ------------------------------
# Run GNG
gng = GrowingNeuralGas(hidden_states_normalized)
gng.fit_network(
    e_b=0.1,       # learning rate for winner
    e_n=0.6,     # learning rate for neighbors
    a_max=5,      # max edge age
    l=100,         # steps between node insertions
    a=0.5,         # error decay on split
    d=0.5,       # error decay on all nodes
    passes=100      # training passes
)

# Extract learned centroids
centroids_ = np.array([gng.graph.nodes[n]['vector'] for n in gng.graph.nodes])

In [66]:
centroids_

array([[1.34439785e-001, 2.24542177e-068, 0.00000000e+000,
        2.20797837e-129, 3.57102068e-001, 2.36786571e-002,
        0.00000000e+000, 4.04540248e-067, 3.58282383e-002,
        0.00000000e+000, 4.95842371e-001, 6.48730510e-068,
        0.00000000e+000, 0.00000000e+000, 4.77178002e-002,
        2.94841388e-001, 0.00000000e+000, 5.82645739e-067,
        4.73546926e-003, 3.12417577e-001, 8.59373455e-002,
        1.50406459e-001, 1.29708182e-001, 1.05176190e-001,
        1.89485076e-128, 1.97779799e-001, 6.20672961e-004,
        1.64351318e-001, 2.42360145e-001, 0.00000000e+000],
       [1.64244592e-001, 1.86957992e-002, 0.00000000e+000,
        1.03448093e-084, 1.54921698e-001, 2.61150124e-002,
        0.00000000e+000, 3.34048927e-001, 5.08995847e-023,
        0.00000000e+000, 7.04472045e-022, 5.33002250e-002,
        0.00000000e+000, 0.00000000e+000, 2.49915279e-004,
        4.18914946e-022, 0.00000000e+000, 4.80819095e-001,
        6.73112704e-024, 9.76779780e-002, 1.46103547e-0

In [67]:
centroids

[tensor([[[0.0000, 0.1769, 0.0000, 0.0000, 1.2324, 0.0000, 0.0000, 1.1984,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 1.5112, 0.0000, 0.0606, 0.4701, 0.5985, 0.0000, 0.0000,
           0.0000, 0.5964, 0.0000, 0.0000, 1.1833, 0.0000]]]),
 tensor([[[6.5218e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00, 9.7842e-02,
           1.0370e-01, 0.0000e+00, 8.2337e-01, 0.0000e+00, 0.0000e+00,
           0.0000e+00, 2.1164e-01, 0.0000e+00, 0.0000e+00, 9.9237e-04,
           0.0000e+00, 0.0000e+00, 1.2749e+00, 0.0000e+00, 3.6242e-01,
           3.8282e-01, 8.0271e-01, 3.7300e-01, 2.4076e-01, 0.0000e+00,
           0.0000e+00, 0.0000e+00, 1.9996e+00, 3.3430e-02, 0.0000e+00]]]),
 tensor([[[0.0000, 0.2667, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.5548,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 1.1919,
           0.0000, 1.3906, 0.0000, 0.1340, 1.7771, 0.7150, 0.0022, 0.8713,
           0.0000, 0.0514, 0.0000, 0.0000, 0.0000, 0.0000

In [68]:
for center_ in centroids_:
    for center in centroids:
        print(distance.cosine(center[0][0].detach().numpy(), center_))

    print('\n')

0.5873594336680945
0.6734022919371252
0.6975601164966704
0.35511396069579726
0.23795318130367704
0.04543524316347625
0.3915624877520095


0.28469195148162163
0.04226758816907861
0.4570205382329001
0.7105977461768926
0.7359663766802342
0.6116960778510374
0.5199626899776695


0.2893155901577291
0.040391520710533424
0.4582893431431243
0.7108520782934622
0.737413866893498
0.6127472749621461
0.5210513707649668


0.28542563156268974
0.041966262674993926
0.4572208638980898
0.710637376183763
0.7361959481177613
0.6118622255079071
0.5201345355484399


0.5873594336680981
0.6734022919371221
0.6975601164966597
0.3551139606956081
0.23795318130356358
0.04543524316355185
0.39156248775197655


0.5882084444324374
0.6732267057374799
0.6963130007466527
0.3290402066789667
0.2226123265331642
0.0568719865983327
0.38741561386431267


0.2298010733811442
0.0690796128515615
0.4432186899645727
0.708487992739814
0.7189279345633641
0.6000374264194572
0.5081716431276284


